In [1]:
import numpy as np
import numba as nb

In [4]:
from numba.cuda import is_available

In [5]:
is_available()

False

In [3]:
from numba import float32, float64, guvectorize, int32, int64, vectorize

In [4]:
gamma = np.random.rand(5,5)

In [5]:
gamma

array([[0.58644248, 0.28263855, 0.35829637, 0.25970216, 0.69097763],
       [0.62179579, 0.34964582, 0.99277127, 0.05929539, 0.59216498],
       [0.76843482, 0.90515028, 0.99980841, 0.85826103, 0.53862432],
       [0.85756994, 0.24818832, 0.95022899, 0.55854304, 0.69176527],
       [0.96375671, 0.74518231, 0.85850394, 0.19561453, 0.17664936]])

In [6]:
@guvectorize(["void(float64[:,:], float64[:,:], float64[:,:])"],
             "(m,n),(n,p)->(m,p)")
def f(a, b, result):
    result = a @ b

/tmp/ipykernel_56183/2231731997.py:4: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  result = a @ b


In [7]:
alpha = np.random.rand(100,50)
beta = np.random.rand(100,50)

In [8]:
f(alpha, beta.T)

array([[ 6.11311479e-310,  4.89864657e-315,  4.89864657e-315, ...,
         4.89217083e-315,  4.89217083e-315,  4.89651031e-315],
       [ 4.89651035e-315,  4.89651039e-315,  4.88806961e-315, ...,
         0.00000000e+000,  4.89769678e-315,  4.89769678e-315],
       [ 4.89769678e-315,  0.00000000e+000,  0.00000000e+000, ...,
         4.89021583e-315,  4.89021587e-315,  4.89021587e-315],
       ...,
       [ 0.00000000e+000,  0.00000000e+000,  0.00000000e+000, ...,
         0.00000000e+000,  0.00000000e+000,  0.00000000e+000],
       [ 0.00000000e+000,  0.00000000e+000,  0.00000000e+000, ...,
         9.88131292e-324,  2.12199579e-314,  4.94065646e-324],
       [ 0.00000000e+000,  0.00000000e+000,  4.24399158e-314, ...,
         0.00000000e+000,  1.69759663e-313, -7.88630858e+303]],
      shape=(100, 100))

In [10]:
echo = np.array([
    [1.,2.,3.],
    [4.,5.,6.],
    [7.,8.,9.]
])

In [25]:
import numba

In [28]:
numba.cuda.detect()

CudaSupportError: Error at driver init: 
Call to cuInit results in CUDA_ERROR_NO_DEVICE (100):

In [23]:
from torch.cuda import is_available
is_available()

ModuleNotFoundError: No module named 'torch'

In [22]:
@guvectorize([(float64[:,:], float64[:])], '(m,n)->(m)', cache=False, target="cuda")
def rowmeans(x, res):
    for i in range(x.shape[0]):
        res[i] = sum(x[i,:])/len(x[i,:])

CudaSupportError: Error at driver init: Call to cuInit results in CUDA_ERROR_NO_DEVICE (100)

In [13]:
alpha

array([[0.69779737, 0.39519283, 0.19492777, ..., 0.11967998, 0.85835705,
        0.06941087],
       [0.4143305 , 0.17072151, 0.46910503, ..., 0.84948882, 0.41769888,
        0.02718304],
       [0.18350991, 0.03281319, 0.96813776, ..., 0.82612264, 0.94690257,
        0.31190969],
       ...,
       [0.01946435, 0.5241222 , 0.50129418, ..., 0.60192655, 0.50117797,
        0.56263857],
       [0.8412726 , 0.72366066, 0.77350961, ..., 0.66982997, 0.50048206,
        0.62811119],
       [0.94748331, 0.88601995, 0.98033912, ..., 0.45151317, 0.14945846,
        0.85057975]], shape=(100, 50))

In [20]:
%%timeit
rowmeans(alpha).reshape(-1,1)

4.45 μs ± 104 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [15]:
%%timeit
np.mean(alpha, axis=1, keepdims=True)

3.99 μs ± 74 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
